# Per-firm commercialization audit review

**Status:** exploratory companion notebook (non-citable)  
**Research question:** B3 — statutory Commercialization Benchmark, §638(qq)(3)  
([docs/research-questions.md](../../docs/research-questions.md))
**Canonical computation:** `scripts/data/audit_one_firm.py` over the benchmark cohort  
CSV (methodology: `docs/commercialization-benchmark-methodology.md`)
**Data as of:** the evaluation-FY cohort CSV's own provenance  

Companion view for reviewing one firm's benchmark verdict as a traceable evidence
bundle with reviewer annotations. Exploratory-tier and non-citable. The live
re-derivation (USAspending API calls, Form D DuckDB matching) stays in
`audit_one_firm.py`; this notebook reads its inputs and records the review.

In [ ]:
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "sbir_etl").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the sbir-analytics checkout")


REPO_ROOT = find_repo_root()
AREA_ID = "nanotechnology"
REPORT_DIR = REPO_ROOT / "data" / "reports" / AREA_ID
RANDOM_SEED = 20260806

EVAL_FY = 2026
BENCHMARK_CSV = REPO_ROOT / "reports" / "validation" / (
    f"commercialization_benchmark_eval_fy{EVAL_FY}.csv"
)
AUDIT_UEI = ""  # set to the UEI under review, e.g. "RMG1AZ1ZH8Q7"

## Data contract

- **Population:** Phase II awardees meeting the §638(qq) tier minimums for the
  evaluation FY (the audited cohort; per the methodology doc the FY2026 audited cohort
  CSV is local-only and not reproducible from `main` yet).
- **Grain:** firm (UEI).
- **Keys:** `uei`; firm display names are not identity.
- **Verdict fields:** `standard_net` / `standard_strict` (§638(mm)) and, for tier1/2
  firms, `increased_net` / `increased_strict` (§638(qq)) statuses with their
  per-Phase-II averages, plus the observed federal and Form D components.
- **Caveats:** `federal_observed` is USAspending-visible obligations only; Form D
  investment is a *lower-bound proxy* for private capital. A FAIL verdict is a
  statement about observable evidence under the frozen windows, not about the firm.

In [ ]:
if not BENCHMARK_CSV.exists():
    print(
        f"Missing {BENCHMARK_CSV.relative_to(REPO_ROOT)} — artifact not present; produce "
        "the audited cohort CSV first (see docs/commercialization-benchmark-methodology.md; "
        "the maintained CLI is scripts/run_benchmark.py)."
    )
    cohort = pd.DataFrame()
else:
    cohort = pd.read_csv(BENCHMARK_CSV)
    print(f"{len(cohort):,} audited firms for FY{EVAL_FY}")
cohort.head()

## Cohort verdict overview

Status distribution across the four verdict variants. `net` vs `strict` disagreement
identifies firms whose PASS depends on how R&D-NAICS federal sales are treated —
exactly the firms worth auditing one at a time.

In [ ]:
if cohort.empty:
    verdicts = pd.DataFrame()
else:
    status_columns = [column for column in cohort.columns if column.endswith("_status")]
    verdicts = pd.DataFrame({column: cohort[column].value_counts(dropna=False) for column in status_columns})
    disagree = cohort[
        cohort.get("standard_net_status", pd.Series(dtype=object)).ne(
            cohort.get("standard_strict_status", pd.Series(dtype=object))
        )
    ]
    print(f"net/strict disagreements (standard tier): {len(disagree)}")
verdicts

## Single-firm evidence bundle

Set `AUDIT_UEI` above and rerun. For the live cross-check (fresh USAspending pulls and
Form D match decisions), run the canonical script — its output is the auditable record:

```bash
uv run python scripts/data/audit_one_firm.py <UEI> --eval-fy 2026
```

In [ ]:
if cohort.empty or not AUDIT_UEI:
    print("Set AUDIT_UEI and ensure the cohort CSV exists to build an evidence bundle.")
    bundle = pd.DataFrame()
else:
    row = cohort[cohort["uei"].eq(AUDIT_UEI)]
    if row.empty:
        print(f"UEI {AUDIT_UEI} not in the FY{EVAL_FY} audited cohort.")
        bundle = pd.DataFrame()
    else:
        bundle = row.T.rename(columns={row.index[0]: AUDIT_UEI})
bundle

## Deterministic review sample

A fixed-seed sample of firms near a verdict boundary, for spot-audits with the
canonical script.

In [ ]:
if cohort.empty:
    review_sample = pd.DataFrame()
else:
    pool = cohort
    if "standard_net_avg_per_p2_usd" in cohort.columns:
        averages = pd.to_numeric(cohort["standard_net_avg_per_p2_usd"], errors="coerce")
        pool = cohort[averages.between(50_000, 200_000)]  # near the $100K/P2 line
    review_sample = pool.sample(min(10, len(pool)), random_state=RANDOM_SEED)
review_sample[[column for column in review_sample.columns if column in (
    "uei", "firm", "state", "federal_observed_usd", "investment_form_d_usd",
    "standard_net_status", "standard_strict_status",
)]] if not review_sample.empty else review_sample

## Reviewer annotations

| UEI | Verdict reviewed | Live cross-check run? | Δ vs CSV | Disposition | Note |
|---|---|---|---|---|---|
| _Draft_ | _standard/increased, net/strict_ | _audit_one_firm.py output attached?_ | _Component deltas_ | _Confirmed / needs rerun_ | _Do not edit the CSV_ |

Discrepancies get fixed by regenerating the cohort CSV through the canonical pipeline,
never by editing values here.